# BirdCLEF+ 2026 EfficientNet-B0

Train a 5-second mel-spectrogram EfficientNet-B0 baseline and generate `submission.csv` in the same Kaggle run. This is the primary competition-safe workflow.


# 1. Train EfficientNet-B0

The model uses primary labels, stratified folds, lightweight augmentation, and single-process audio loading for stable Kaggle execution.


## 1.1 Environment


In [1]:
from pathlib import Path
import json
import os
import random
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 80)

# Competition reruns should never depend on external downloads.
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"


class CFG:
    seed = 42
    competition_name = "birdclef-2026"
    data_root = None
    artifact_dir = Path("/kaggle/working/artifacts")


def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)


def find_data_root() -> Path:
    candidates = [
        Path("/kaggle/input/birdclef-2026"),
        Path("/kaggle/input/birdclef-2026-repack/birdclef-2026"),
        Path("/kaggle/input/birdclef-2026-repack"),
    ]
    for path in candidates:
        if (path / "train.csv").exists():
            return path
    input_root = Path("/kaggle/input")
    if input_root.exists():
        matches = list(input_root.glob("**/train.csv"))
        if matches:
            return matches[0].parent
    raise FileNotFoundError("Could not find train.csv. Attach the BirdCLEF 2026 dataset.")


seed_everything(CFG.seed)
CFG.data_root = find_data_root()
CFG.artifact_dir.mkdir(parents=True, exist_ok=True)

print(f"Data root: {CFG.data_root}")
print(f"Output directory: {CFG.artifact_dir}")

try:
    import timm
except ImportError:
    import sys
    !{sys.executable} -m pip install -q timm
    import timm

import librosa
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

torch.manual_seed(CFG.seed)
torch.cuda.manual_seed_all(CFG.seed)
torch.backends.cudnn.benchmark = True
torch.set_num_threads(min(2, os.cpu_count() or 1))
try:
    torch.multiprocessing.set_sharing_strategy("file_system")
except RuntimeError:
    pass


class CFG(CFG):
    artifact_dir = Path("/kaggle/working/artifacts/effnet_b0")
    sample_rate = 32000
    duration = 5.0
    n_fft = 2048
    hop_length = 512
    n_mels = 128
    fmin = 20
    fmax = 16000
    n_splits = 5
    fold = 0
    backbone = "efficientnet_b0"
    pretrained = False
    pretrained_weight_path = None  # Example: Path("/kaggle/input/effnet-b0-timm-weights/model.safetensors")
    epochs = 5
    batch_size = 32
    # Kaggle notebooks can crash with multiprocessing DataLoader workers when
    # audio decoding happens inside __getitem__. Keep this at 0 for stability.
    num_workers = 0
    force_single_process_loader = True
    lr = 3e-4
    weight_decay = 1e-2
    label_smoothing = 0.05
    max_train_samples = None
    max_valid_samples = None


CFG.artifact_dir.mkdir(parents=True, exist_ok=True)
TRAIN_ARTIFACT_DIR = CFG.artifact_dir
SUBMISSION_ARTIFACT_DIR = Path("/kaggle/working/artifacts/submission")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")


Data root: /kaggle/input/competitions/birdclef-2026
Output directory: /kaggle/working/artifacts
Device: cuda


## 1.2 Metadata And Folds


Build the label map and validation fold from `train.csv`. The fold file is saved with the artifacts so model results can be traced back to the exact split.


In [2]:
train = pd.read_csv(CFG.data_root / "train.csv")
train["filepath"] = train["filename"].map(lambda x: CFG.data_root / "train_audio" / x)
labels = sorted(train["primary_label"].unique())
label_to_idx = {label: idx for idx, label in enumerate(labels)}
idx_to_label = {idx: label for label, idx in label_to_idx.items()}
train["target"] = train["primary_label"].map(label_to_idx)

train["fold"] = -1
group_col = "filename"
try:
    splitter = StratifiedGroupKFold(n_splits=CFG.n_splits, shuffle=True, random_state=CFG.seed)
    splits = splitter.split(train, train["target"], groups=train[group_col])
except ValueError:
    splitter = StratifiedKFold(n_splits=CFG.n_splits, shuffle=True, random_state=CFG.seed)
    splits = splitter.split(train, train["target"])

for fold, (_, valid_idx) in enumerate(splits):
    train.loc[valid_idx, "fold"] = fold

train.to_csv(CFG.artifact_dir / "train_folds.csv", index=False)
(CFG.artifact_dir / "labels.json").write_text(json.dumps(idx_to_label, indent=2), encoding="utf-8")

train_df = train[train["fold"] != CFG.fold].reset_index(drop=True)
valid_df = train[train["fold"] == CFG.fold].reset_index(drop=True)
if CFG.max_train_samples:
    train_df = train_df.sample(CFG.max_train_samples, random_state=CFG.seed).reset_index(drop=True)
if CFG.max_valid_samples:
    valid_df = valid_df.sample(CFG.max_valid_samples, random_state=CFG.seed).reset_index(drop=True)

print(f"Train rows: {len(train_df):,}")
print(f"Valid rows: {len(valid_df):,}")
print(f"Classes: {len(labels):,}")


Train rows: 28,439
Valid rows: 7,110
Classes: 206


## 1.3 Dataset


Each recording becomes a normalized mel-spectrogram. Training uses random gain augmentation; validation uses deterministic 5-second crops for a stable reference metric.


In [3]:
def load_audio(path: Path, duration: float, train_mode: bool) -> np.ndarray:
    target_len = int(CFG.sample_rate * duration)
    offset = 0.0
    y, _ = librosa.load(path, sr=CFG.sample_rate, mono=True, offset=offset, duration=duration)
    if len(y) < target_len:
        y = np.pad(y, (0, target_len - len(y)))
    return y[:target_len].astype(np.float32)


def audio_to_mel(y: np.ndarray) -> np.ndarray:
    mel = librosa.feature.melspectrogram(
        y=y,
        sr=CFG.sample_rate,
        n_fft=CFG.n_fft,
        hop_length=CFG.hop_length,
        n_mels=CFG.n_mels,
        fmin=CFG.fmin,
        fmax=CFG.fmax,
        power=2.0,
    )
    mel = librosa.power_to_db(mel, ref=np.max)
    mel = (mel - mel.mean()) / (mel.std() + 1e-6)
    return mel.astype(np.float32)


class BirdDataset(Dataset):
    def __init__(self, df: pd.DataFrame, train_mode: bool):
        self.df = df.reset_index(drop=True)
        self.train_mode = train_mode

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        y = load_audio(row["filepath"], CFG.duration, self.train_mode)
        if self.train_mode and random.random() < 0.5:
            y = y * random.uniform(0.75, 1.25)
        x = torch.from_numpy(audio_to_mel(y)).unsqueeze(0)
        target = torch.tensor(row["target"], dtype=torch.long)
        return x, target


def effective_num_workers() -> int:
    if CFG.force_single_process_loader or Path("/kaggle/working").exists():
        return 0
    return int(CFG.num_workers)


def make_loader(dataset: Dataset, batch_size: int, shuffle: bool) -> DataLoader:
    workers = effective_num_workers()
    loader_kwargs = {
        "batch_size": batch_size,
        "shuffle": shuffle,
        "num_workers": workers,
        "pin_memory": device.type == "cuda",
        "drop_last": False,
    }
    if workers > 0:
        loader_kwargs["persistent_workers"] = True
        loader_kwargs["prefetch_factor"] = 2
    return DataLoader(dataset, **loader_kwargs)


print(f"Effective DataLoader workers: {effective_num_workers()}")
train_loader = make_loader(BirdDataset(train_df, train_mode=True), CFG.batch_size, True)
valid_loader = make_loader(BirdDataset(valid_df, train_mode=False), CFG.batch_size * 2, False)


Effective DataLoader workers: 0


## 1.4 Model


EfficientNet-B0 is small enough for fast Kaggle reruns while still giving a useful acoustic CNN baseline for comparison against Perch features. External downloads are disabled; attach a local Kaggle input checkpoint only if pretrained initialization is needed.


In [4]:
def torch_load(path: Path):
    if path.suffix == ".safetensors":
        from safetensors.torch import load_file
        return load_file(path, device=str(device))
    try:
        return torch.load(path, map_location=device, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=device)


def unwrap_state_dict(checkpoint):
    if isinstance(checkpoint, dict):
        for key in ["state_dict", "model", "model_state_dict"]:
            if key in checkpoint and isinstance(checkpoint[key], dict):
                return checkpoint[key]
    return checkpoint


def load_local_pretrained_weights(model: nn.Module, path: Path) -> None:
    state_dict = unwrap_state_dict(torch_load(path))
    model_state = model.state_dict()
    filtered, skipped = {}, []

    for key, value in state_dict.items():
        key = key.removeprefix("module.").removeprefix("model.")
        if key not in model_state:
            skipped.append(key)
            continue
        if value.shape == model_state[key].shape:
            filtered[key] = value
        elif value.ndim == 4 and model_state[key].ndim == 4 and value.shape[1] == 3 and model_state[key].shape[1] == 1:
            filtered[key] = value.mean(dim=1, keepdim=True)
        else:
            skipped.append(key)

    missing, unexpected = model.load_state_dict(filtered, strict=False)
    print(f"Loaded local pretrained tensors: {len(filtered):,} from {path}")
    print(f"Skipped incompatible tensors: {len(skipped):,}; missing after load: {len(missing):,}; unexpected: {len(unexpected):,}")


class BirdClassifier(nn.Module):
    def __init__(self, num_classes: int):
        super().__init__()
        self.model = timm.create_model(
            CFG.backbone,
            pretrained=False,
            in_chans=1,
            num_classes=num_classes,
        )

    def forward(self, x):
        return self.model(x)


model = BirdClassifier(num_classes=len(labels)).to(device)
if CFG.pretrained_weight_path is not None:
    load_local_pretrained_weights(model, Path(CFG.pretrained_weight_path))
criterion = nn.CrossEntropyLoss(label_smoothing=CFG.label_smoothing)
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.lr, weight_decay=CFG.weight_decay)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG.epochs)
scaler = torch.cuda.amp.GradScaler(enabled=device.type == "cuda")


In [5]:
def train_one_epoch() -> float:
    model.train()
    total_loss = 0.0
    for x, y in tqdm(train_loader, desc="train", leave=False):
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=scaler.is_enabled()):
            loss = criterion(model(x), y)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * x.size(0)
    return total_loss / max(len(train_loader.dataset), 1)


@torch.no_grad()
def validate() -> tuple[float, float]:
    model.eval()
    total_loss = 0.0
    correct = 0
    seen = 0
    for x, y in tqdm(valid_loader, desc="valid", leave=False):
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        logits = model(x)
        loss = criterion(logits, y)
        total_loss += loss.item() * x.size(0)
        correct += (logits.argmax(dim=1) == y).sum().item()
        seen += x.size(0)
    return total_loss / max(seen, 1), correct / max(seen, 1)


## 1.5 Training


Save the checkpoint with the best validation accuracy and record the epoch-level learning curve in `history.csv`.


In [6]:
history = []
best_acc = 0.0

for epoch in range(1, CFG.epochs + 1):
    train_loss = train_one_epoch()
    valid_loss, valid_acc = validate()
    scheduler.step()
    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "valid_loss": valid_loss,
        "valid_acc": valid_acc,
        "lr": scheduler.get_last_lr()[0],
    }
    history.append(row)
    print(row)

    if valid_acc > best_acc:
        best_acc = valid_acc
        torch.save(
            {
                "model": model.state_dict(),
                "label_to_idx": label_to_idx,
                "cfg": {k: v for k, v in CFG.__dict__.items() if not k.startswith("_")},
                "valid_acc": best_acc,
            },
            CFG.artifact_dir / "best_effnet_b0.pt",
        )

history_df = pd.DataFrame(history)
history_df.to_csv(CFG.artifact_dir / "history.csv", index=False)
print(f"Best valid accuracy: {best_acc:.4f}")
print(f"Saved outputs to {CFG.artifact_dir}")


train:   0%|          | 0/889 [00:00<?, ?it/s]

valid:   0%|          | 0/112 [00:00<?, ?it/s]

{'epoch': 1, 'train_loss': 4.334321711476608, 'valid_loss': 3.5051888614096556, 'valid_acc': 0.24627285513361463, 'lr': 0.0002713525491562421}


train:   0%|          | 0/889 [00:00<?, ?it/s]

valid:   0%|          | 0/112 [00:00<?, ?it/s]

...

valid:   0%|          | 0/112 [00:00<?, ?it/s]

{'epoch': 4, 'train_loss': 1.8974975918813062, 'valid_loss': 2.45678634388705, 'valid_acc': 0.5225035161744023, 'lr': 2.8647450843757894e-05}


train:   0%|          | 0/889 [00:00<?, ?it/s]

valid:   0%|          | 0/112 [00:00<?, ?it/s]

{'epoch': 5, 'train_loss': 1.4742311517742683, 'valid_loss': 2.4469982999957227, 'valid_acc': 0.5272855133614627, 'lr': 0.0}
Best valid accuracy: 0.5273
Saved outputs to /kaggle/working/artifacts/effnet_b0


## 1.6 Save Training Outputs


Save the checkpoint, label map, fold file, and training history as a compact Kaggle output bundle.


In [7]:
import zipfile
from IPython.display import FileLink

zip_path = Path("/kaggle/working/birdclef_effnet_b0_artifacts.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(CFG.artifact_dir.rglob("*")):
        if path.is_file():
            zf.write(path, arcname=path.relative_to(CFG.artifact_dir.parent))

print(f"Wrote {zip_path} ({zip_path.stat().st_size / 1024 / 1024:.2f} MB)")
display(FileLink(zip_path))


Wrote /kaggle/working/birdclef_effnet_b0_artifacts.zip (16.51 MB)


/kaggle/working/birdclef_effnet_b0_artifacts.zip

# 2. Generate Submission

Use the checkpoint trained above to score the test soundscape windows expected by `sample_submission.csv`.


## 2.1 Inference Environment


In [8]:
import zipfile
from IPython.display import FileLink

SUBMISSION_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
CFG.inference_batch_size = 32

checkpoint_path = TRAIN_ARTIFACT_DIR / "best_effnet_b0.pt"
labels_path = TRAIN_ARTIFACT_DIR / "labels.json"

if not checkpoint_path.exists() or not labels_path.exists():
    raise FileNotFoundError("Training outputs are missing. Run the training cells before submission inference.")

print(f"Device: {device}")
print(f"Checkpoint: {checkpoint_path}")
print(f"Labels: {labels_path}")
print(f"Submission output directory: {SUBMISSION_ARTIFACT_DIR}")


Device: cuda
Checkpoint: /kaggle/working/artifacts/effnet_b0/best_effnet_b0.pt
Labels: /kaggle/working/artifacts/effnet_b0/labels.json
Submission output directory: /kaggle/working/artifacts/submission


## 2.2 Competition Files And Checkpoint


Load `sample_submission.csv`, `best_effnet_b0.pt`, and `labels.json` from the current Kaggle run. If the checkpoint is missing, rerun the training section first.


In [9]:
sample_path = CFG.data_root / "sample_submission.csv"
sample_submission = pd.read_csv(sample_path)
idx_to_label = {int(k): v for k, v in json.loads(labels_path.read_text()).items()}
labels = [idx_to_label[i] for i in sorted(idx_to_label)]
label_to_idx = {label: i for i, label in enumerate(labels)}
target_columns = [c for c in sample_submission.columns if c != "row_id"]
target_positions = [i for i, label in enumerate(labels) if label in target_columns]
target_labels = [labels[i] for i in target_positions]

print(f"Sample submission: {sample_path}")
print(f"Rows: {len(sample_submission):,}")
print(f"Submission target columns: {len(target_columns):,}")
print(f"Model classes: {len(labels):,}")
display(sample_submission.head())


Sample submission: /kaggle/input/competitions/birdclef-2026/sample_submission.csv
Rows: 3
Submission target columns: 234
Model classes: 206


,row_id,1161364,116570,1176823,1491113,1595929,209233,22930,22956,22961,22967,22973,22983,22985,23150,23154,23158,23176,23724,24279,24285,24287,24321,244024,25073,25092,25214,326272,41970,43435,47144,47158son01,47158son02,47158son03,47158son04,47158son05,47158son06,47158son07,47158son08,47158son09,...,sobtyr1,socfly1,sofspi1,souant1,soulap1,souscr1,spbant3,spispi1,sptnig1,squcuc1,stbwoo2,strcuc1,strher2,strowl1,swthum1,swtman1,tattin1,thlwre1,toctou1,trokin,trsowl,undtin1,varant1,watjac1,wesfie1,wfwduc1,whbant2,whbwar2,whiwoo1,whlspi1,whnjay1,whtdov,whwpic1,y00678,yebcar,yebela1,yecmac,yecpar,yehcar1,yeofly1
0,BC2026_Test_0001_S05_20250227_010002_5,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,...,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274
1,BC2026_Test_0001_S05_20250227_010002_10,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,...,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274
2,BC2026_Test_0001_S05_20250227_010002_15,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,...,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274


## 2.3 Model And Audio Index


In [10]:
checkpoint = torch_load(checkpoint_path)
model = BirdClassifier(num_classes=len(labels)).to(device)
model.load_state_dict(checkpoint["model"])
model.eval()


def row_to_stem_and_end_time(row_id: str) -> tuple[str, float]:
    stem, sep, end = str(row_id).rpartition("_")
    if sep and end.replace(".", "", 1).isdigit():
        return stem, float(end)
    return str(row_id), CFG.duration


def build_audio_index() -> dict[str, Path]:
    audio_index = {}
    for folder in [CFG.data_root / "test_soundscapes", CFG.data_root / "test_audio"]:
        if folder.exists():
            for ext in ("*.ogg", "*.wav", "*.flac", "*.mp3"):
                for path in folder.glob(ext):
                    audio_index[path.stem] = path
    return audio_index


audio_index = build_audio_index()
print(f"Indexed audio files: {len(audio_index):,}")


Indexed audio files: 0


## 2.4 Inference


Parse each `row_id` as a 5-second window ending at its timestamp, convert the segment to a mel-spectrogram, and write class probabilities into the submission columns.


In [11]:
def load_audio_segment(path: Path, end_time: float) -> np.ndarray:
    offset = max(0.0, float(end_time) - CFG.duration)
    target_len = int(CFG.sample_rate * CFG.duration)
    y, _ = librosa.load(path, sr=CFG.sample_rate, mono=True, offset=offset, duration=CFG.duration)
    if len(y) < target_len:
        y = np.pad(y, (0, target_len - len(y)))
    return y[:target_len].astype(np.float32)


def predict_batch(batch: list[np.ndarray]) -> np.ndarray:
    x = torch.from_numpy(np.stack(batch)).unsqueeze(1).to(device)
    with torch.no_grad():
        logits = model(x)
        return torch.softmax(logits, dim=1).cpu().numpy()


submission = sample_submission.copy()
for col in target_columns:
    submission[col] = 0.0

if len(audio_index) == 0 and len(submission) <= 3:
    print("Tiny public dry-run detected with no test audio; preserving sample submission values.")
    submission = sample_submission.copy()
else:
    batch, batch_rows = [], []
    for row_idx, row_id in tqdm(list(enumerate(submission["row_id"])), desc="inference"):
        stem, end_time = row_to_stem_and_end_time(row_id)
        audio_path = audio_index.get(stem)
        if audio_path is None:
            raise FileNotFoundError(f"Missing audio for row_id={row_id}; expected stem={stem}")
        batch.append(audio_to_mel(load_audio_segment(audio_path, end_time)))
        batch_rows.append(row_idx)
        if len(batch) == CFG.inference_batch_size:
            probs = predict_batch(batch)[:, target_positions]
            submission.loc[batch_rows, target_labels] = probs
            batch, batch_rows = [], []

    if batch:
        probs = predict_batch(batch)[:, target_positions]
        submission.loc[batch_rows, target_labels] = probs

display(submission.head())


Tiny public dry-run detected with no test audio; preserving sample submission values.


,row_id,1161364,116570,1176823,1491113,1595929,209233,22930,22956,22961,22967,22973,22983,22985,23150,23154,23158,23176,23724,24279,24285,24287,24321,244024,25073,25092,25214,326272,41970,43435,47144,47158son01,47158son02,47158son03,47158son04,47158son05,47158son06,47158son07,47158son08,47158son09,...,sobtyr1,socfly1,sofspi1,souant1,soulap1,souscr1,spbant3,spispi1,sptnig1,squcuc1,stbwoo2,strcuc1,strher2,strowl1,swthum1,swtman1,tattin1,thlwre1,toctou1,trokin,trsowl,undtin1,varant1,watjac1,wesfie1,wfwduc1,whbant2,whbwar2,whiwoo1,whlspi1,whnjay1,whtdov,whwpic1,y00678,yebcar,yebela1,yecmac,yecpar,yehcar1,yeofly1
0,BC2026_Test_0001_S05_20250227_010002_5,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,...,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274
1,BC2026_Test_0001_S05_20250227_010002_10,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,...,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274
2,BC2026_Test_0001_S05_20250227_010002_15,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,...,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274


## 2.5 Submission Checks

- `submission.csv` should match the sample submission shape.
- Hidden-test runs should index all expected soundscape files.
- Validation accuracy is a sanity check, not a leaderboard guarantee.


## 2.6 Save Submission


In [12]:
submission_path = Path("/kaggle/working/submission.csv")
submission.to_csv(submission_path, index=False)
submission.to_csv(SUBMISSION_ARTIFACT_DIR / "submission.csv", index=False)

zip_path = Path("/kaggle/working/birdclef_effnet_b0_submission.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(submission_path, arcname="submission.csv")

print(f"Wrote submission: {submission_path}")
print(f"Wrote submission zip: {zip_path} ({zip_path.stat().st_size / 1024 / 1024:.2f} MB)")
display(FileLink(submission_path))
display(FileLink(zip_path))


Wrote submission: /kaggle/working/submission.csv
Wrote submission zip: /kaggle/working/birdclef_effnet_b0_submission.zip (0.00 MB)


/kaggle/working/submission.csv

/kaggle/working/birdclef_effnet_b0_submission.zip